In [ ]:
# ==============================================================================
# 1. INITIALISATION ET VECTORISATION DES DONNÉES
# ==============================================================================
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import datasets, models, layers, optimizers

# Configuration des graines pour la reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

# Configuration globale
NUM_WORDS = 10000
BATCH_SIZE = 512
EPOCHS = 20

print("--- Chargement des données IMDB ---")
# Chargement des données intégrées à Keras
(train_data, train_labels), (test_data, test_labels) = datasets.imdb.load_data(num_words=NUM_WORDS)

# Fonction de vectorisation (One-Hot Encoding manuel)
def vectorize_sequences(sequences, dimension=NUM_WORDS):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0
    return results

# Transformation des listes d'entiers en matrices binaires (Tensors de fait)
X_train_full = vectorize_sequences(train_data)
X_test = vectorize_sequences(test_data)

# Conversion des étiquettes en float32
y_train_full = np.asarray(train_labels).astype('float32')
y_test = np.asarray(test_labels).astype('float32')

# Création d'un jeu de validation (10 000 premiers échantillons)
X_val = X_train_full[:10000]
X_train = X_train_full[10000:]

y_val = y_train_full[:10000]
y_train = y_train_full[10000:]

print(f"Données d'entraînement : {X_train.shape}")
print(f"Données de validation  : {X_val.shape}")
print(f"Données de test        : {X_test.shape}")

# ==============================================================================
# 2. CONSTRUCTION DU MODÈLE FEEDFORWARD (MLP)
# ==============================================================================
def create_dense_model():
    model = models.Sequential([
        # Couche cachée 1 : 16 unités, activation ReLU
        layers.Dense(16, activation='relu', input_shape=(NUM_WORDS,)),
        # Couche cachée 2 : 16 unités, activation ReLU
        layers.Dense(16, activation='relu'),
        # Couche de sortie : 1 unité, activation Sigmoid (Probabilité binaire)
        layers.Dense(1, activation='sigmoid')
    ])
    return model

model = create_dense_model()

# Compilation avec RMSprop, binary_crossentropy et la métrique accuracy
model.compile(
    optimizer=optimizers.RMSprop(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ==============================================================================
# 3. PREMIER ENTRAÎNEMENT ET VISUALISATION (DÉTECTION DE L'OVERFITTING)
# ==============================================================================
print("\n--- Premier entraînement (20 époques) ---")
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    verbose=1
)

# Récupération des métriques pour les graphiques
history_dict = history.history
loss_vals = history_dict['loss']
val_loss_vals = history_dict['val_loss']
acc_vals = history_dict['accuracy']
val_acc_vals = history_dict['val_accuracy']
epochs_range = range(1, len(loss_vals) + 1)

# Graphique de la Perte (Loss)
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, loss_vals, 'bo', label='Perte Entraînement')
plt.plot(epochs_range, val_loss_vals, 'b', label='Perte Validation')
plt.title('Perte pendant l\'entraînement et la validation')
plt.xlabel('Époques')
plt.ylabel('Perte (Loss)')
plt.legend()

# Graphique de l'Exactitude (Accuracy)
plt.subplot(1, 2, 2)
plt.plot(epochs_range, acc_vals, 'ro', label='Précision Entraînement')
plt.plot(epochs_range, val_acc_vals, 'r', label='Précision Validation')
plt.title('Exactitude pendant l\'entraînement et la validation')
plt.xlabel('Époques')
plt.ylabel('Précision (Accuracy)')
plt.legend()

plt.tight_layout()
plt.show()

# En observant les courbes, vous constaterez que la perte de validation 
# commence à remonter généralement vers la 4ème ou 5ème époque : c'est le pic d'overfitting.

# ==============================================================================
# 4.RÉENTRAÎNEMENT OPTIMAL ET ÉVALUATION FINALE
# ==============================================================================
# On détermine l'époque optimale d'après le graphique (souvent l'époque 4)
OPTIMAL_EPOCHS = 4 

print(f"\n--- Réentraînement du modèle final avec {OPTIMAL_EPOCHS} époques ---")
final_model = create_dense_model()
final_model.compile(
    optimizer=optimizers.RMSprop(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# On entraîne cette fois sur l'ensemble complet d'entraînement (Train + Val combinés)
final_model.fit(
    X_train_full, y_train_full,
    epochs=OPTIMAL_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

print("\n--- Évaluation finale sur le jeu de test (Données jamais vues) ---")
test_loss, test_accuracy = final_model.evaluate(X_test, y_test)
print(f"Perte finale sur le Test : {test_loss:.4f}")
print(f"Précision finale sur le Test : {test_accuracy:.4f}")